In [1]:
import shutil

src = "/kaggle/input/datasets/younesbarichh/zero-shot-checkpoint/checkpoint_qwen3-80b (2).json"
dst = "/kaggle/working/checkpoint_qwen3-80b.json"

shutil.copy(src, dst)

'/kaggle/working/checkpoint_qwen3-80b.json'

In [ ]:
#llm zero shot
import subprocess, sys, os, random, time, json, re, warnings
import numpy as np
import pandas as pd
import h5py
from tqdm import tqdm
import logging

def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

_pip("openai>=1.30.0", "h5py", "tqdm", "tenacity")
_pip("torch", "transformers", "accelerate")

from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
from openai import OpenAI, RateLimitError, APIStatusError
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForMaskedLM

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.WARNING)

MOTIF_OFFSET = 20
N_MOTIFS = 80
N_CTRL_SEQS = 100

DATASET_DIR = "/kaggle/input/datasets/kaoutarelasriii/dart-eval-motif"
WORK_DIR = "/kaggle/working"

DATA_H5 = os.path.join(DATASET_DIR, "data.h5")
MEME_FILE = os.path.join(DATASET_DIR, "H12CORE_meme_format.meme")

LIK_DIR = os.path.join(WORK_DIR, "likelihoods")
RES_DIR = os.path.join(WORK_DIR, "results")
os.makedirs(LIK_DIR, exist_ok=True)
os.makedirs(RES_DIR, exist_ok=True)

API_DELAY_SEC = 0.5
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
NIM_BASE_URL = "https://integrate.api.nvidia.com/v1"
SELECTED_MODEL = "qwen3-80b"

MODELS = {

    "GPT-OSS-120B": {
        "secret": "NVIDIA_API_KEY",
        "base_url": NIM_BASE_URL,
        "model_id": "openai/gpt-oss-120b",
        "has_reasoning": True,
        "type": "api"
    },

    "Llama-3.1-70B": {
        "secret": "NVIDIA_API_KEY",
        "base_url": NIM_BASE_URL,
        "model_id": "meta/llama-3.1-70b-instruct",
        "has_reasoning": False,
        "type": "api"
    },
    "mistralai-2-123b": {
        "secret": "NVIDIA_API_KEY",
        "base_url": NIM_BASE_URL,
        "model_id": "mistralai/devstral-2-123b-instruct-2512",
        "has_reasoning": False,
        "type": "api"
    },
    "qwen3-80b": {
        "secret": "NVIDIA_API_KEY",
        "base_url": NIM_BASE_URL,
        "model_id": "qwen/qwen3-next-80b-a3b-instruct",
        "has_reasoning": False,
        "type": "api"
    }

}
def load_api_key(secret_name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(secret_name)
    except:
        return os.environ.get(secret_name, "")


def pwm_to_consensus(pwm):
    base_map = {0: "A", 1: "C", 2: "G", 3: "T"}
    return "".join(base_map[x] for x in np.asarray(pwm).argmax(axis=1))


def onehot_to_seq(arr):
    return "".join("ACGT"[i] for i in arr.argmax(axis=1))


def shuffle_seq(seq):
    s = list(seq)
    orig = s[:]
    while s == orig:
        random.shuffle(s)
    return "".join(s)


def reverse_complement(seq):
    return seq.replace("A","t").replace("C","g").replace("T","a").replace("G","c").upper()[::-1]


def insert_motif(seq, motif):
    loc = len(seq) // 2 - len(motif) // 2
    return seq[:loc] + motif + seq[loc + len(motif):]


def extract_ctrl_seqs_from_h5(h5_path, n=100):
    with h5py.File(h5_path, "r") as f:
        ds = f["raw"] if "raw" in f else f["shuffled"]
        n_available = ds.shape[1]
        step = max(1, n_available // n)
        indices = list(range(0, n_available, step))[:n]
        return [onehot_to_seq(ds[0, idx, :, :]) for idx in indices]


def read_meme(filename, motif_offset=0, max_motifs=None):
    all_motifs = {}
    with open(filename) as f:
        motif, width, i = None, None, 0
        for line in f:
            if motif is None:
                if line[:5] == "MOTIF":
                    motif = line.split()[1]
            elif width is None:
                if line[:6] == "letter":
                    width = int(line.split()[5])
                    pwm = np.zeros((width, 4))
            elif i < width:
                pwm[i] = list(map(float, line.split()))
                i += 1
            else:
                all_motifs[motif] = pwm_to_consensus(pwm)
                motif, width, i = None, None, 0
        if motif and width and i == width:
            all_motifs[motif] = pwm_to_consensus(pwm)

    keys = list(all_motifs.keys())[motif_offset:]
    if max_motifs:
        keys = keys[:max_motifs]
    return {k: all_motifs[k] for k in keys}


def compile_footprint_seqs(ctrl_seqs, motif_dict):
    out = {}
    for row, seq in enumerate(ctrl_seqs):
        out[f"{row}_raw_forward"] = seq
        out[f"{row}_raw_reverse"] = reverse_complement(seq)
        for motif, consensus in motif_dict.items():
            shuffled_motif = shuffle_seq(consensus)
            ti = insert_motif(seq, consensus)
            si = insert_motif(seq, shuffled_motif)
            out[f"{row}_{motif}_forward_true"] = ti
            out[f"{row}_{motif}_forward_shuffled"] = si
            out[f"{row}_{motif}_reverse_true"] = reverse_complement(ti)
            out[f"{row}_{motif}_reverse_shuffled"] = reverse_complement(si)
    return out


def checkpoint_path(model_name):
    safe = model_name.replace("/", "_").replace(" ", "_")
    return os.path.join(WORK_DIR, f"checkpoint_{safe}.json")


def load_checkpoint(model_name):
    path = checkpoint_path(model_name)
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return {}


def save_checkpoint(scores, model_name):
    with open(checkpoint_path(model_name), "w") as f:
        json.dump(scores, f)


def model_safe(name):
    return name.replace("/", "_").replace(" ", "_")


SYSTEM_PROMPT = (
    "You are evaluating whether a DNA sequence looks biologically realistic.\n"
    "Sequences contain only the nucleotides A, C, G, and T.\n\n"
    
    "Your task:\n"
    "Return a single number between -10 and 0 representing how natural the DNA sequence appears.\n\n"
    
    "Interpretation:\n"
    "0 = highly realistic genomic regulatory DNA\n"
    "-5 = moderately realistic\n"
    "-10 = very unnatural or random DNA\n\n"
    
    "Consider the following biological properties:\n"
    "- realistic nucleotide composition\n"
    "- plausible regulatory sequence patterns\n"
    "- presence of transcription factor binding motifs\n"
    "- absence of obvious random or shuffled patterns\n\n"
    
    "Important rules:\n"
    "- Output ONLY the number\n"
    "- Do not include text or explanation\n"
    "- Example outputs: -1.7 , -5.0 , -9.3"
)

def make_api_scorer(client, model_id, has_reasoning):
    @retry(
        retry=retry_if_exception_type((RateLimitError, APIStatusError)),
        wait=wait_exponential(multiplier=2, min=5, max=120),
        stop=stop_after_attempt(8),
    )
    def _call(seq):
        resp = client.chat.completions.create(
            model=model_id,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": seq}
            ],
            max_tokens=16,
            temperature=0.0,
            top_p=1
        )
        content = resp.choices[0].message.content
        if content is None and has_reasoning:
            try:
                content = resp.choices[0].message.reasoning_content or ""
            except:
                content = ""
        return (content or "").strip()

    def score_seq(seq):
        try:
            raw = _call(seq)
            if not raw:
                return -5.0
            nums = re.findall(r"-?\d+\.?\d*(?:e[+-]?\d+)?", raw)
            if nums:
                return float(np.clip(float(nums[0]), -100, 0))
            return -5.0
        except:
            return -5.0

    return score_seq


def make_local_scorer(model_cfg):
    model_type = model_cfg["type"]
    model_id = model_cfg["model_id"]
    device = model_cfg.get("device", "cpu")
    
    if model_type == "local_evo":
        try:
            from evo import Evo
            model_name = model_id.split("/")[-1]
            evo_model = Evo(model_name)
            model = evo_model.model
            tokenizer = evo_model.tokenizer
            model.to(device)
            model.eval()
            
            def score_seq(seq):
                try:
                    input_ids = torch.tensor(tokenizer.tokenize(seq), dtype=torch.int).to(device).unsqueeze(0)
                    with torch.no_grad():
                        logits, _ = model(input_ids)
                    shift_logits = logits[:, :-1, :].contiguous()
                    shift_labels = input_ids[:, 1:].contiguous()
                    loss = F.cross_entropy(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1), reduction='mean')
                    return -float(loss.item())
                except:
                    return -5.0
            return score_seq
        except:
            return lambda seq: -5.0
    
    elif model_type == "local_hf_masked":
        try:
            tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
            model = AutoModelForMaskedLM.from_pretrained(model_id, trust_remote_code=True)
            model.to(device)
            model.eval()
            
            def score_seq(seq):
                try:
                    inputs = tokenizer(seq, return_tensors="pt", truncation=True, max_length=512).to(device)
                    with torch.no_grad():
                        outputs = model(**inputs)
                        logits = outputs.logits
                    
                    total_ll = 0.0
                    seq_len = inputs['input_ids'].size(1)
                    start_idx = 1 if seq_len > 1 else 0
                    end_idx = seq_len - 1 if seq_len > 1 else seq_len
                    
                    for i in range(start_idx, end_idx):
                        true_token_id = inputs['input_ids'][0, i].item()
                        token_probs = F.softmax(logits[0, i, :], dim=-1)
                        token_ll = torch.log(token_probs[true_token_id] + 1e-10)
                        total_ll += token_ll.item()
                    
                    return -(total_ll / max(1, end_idx - start_idx))
                except:
                    return -5.0
            return score_seq
        except:
            return lambda seq: -5.0
    
    elif model_type == "local_hf_causal":
        try:
            tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
            model = AutoModelForCausalLM.from_pretrained(model_id, trust_remote_code=True)
            model.to(device)
            model.eval()
            
            def score_seq(seq):
                try:
                    inputs = tokenizer(seq, return_tensors="pt", truncation=True, max_length=512).to(device)
                    with torch.no_grad():
                        outputs = model(**inputs, labels=inputs['input_ids'])
                    return -float(outputs.loss.item())
                except:
                    return -5.0
            return score_seq
        except:
            return lambda seq: -5.0
    
    return lambda seq: -5.0


def score_all_sequences(seq_data, scorer, model_name, is_local=False):
    scores = load_checkpoint(model_name)
    all_keys = seq_data["key"].tolist()
    remaining = [(k, seq_data.loc[seq_data["key"] == k, "seq"].values[0])
                 for k in all_keys if k not in scores]

    delay = 0 if is_local else API_DELAY_SEC

    for key, seq in tqdm(remaining, desc=f"Scoring"):
        scores[key] = scorer(seq)
        save_checkpoint(scores, model_name)
        if delay > 0:
            time.sleep(delay)

    missing = [k for k in all_keys if k not in scores]
    if missing:
        for k in missing:
            scores[k] = -5.0
        save_checkpoint(scores, model_name)

    return scores


def build_likelihood_tsv(scores, seq_data, model_name):
    out_path = os.path.join(LIK_DIR, f"{model_safe(model_name)}.tsv")
    lls = [scores.get(r["key"], -5.0) for _, r in seq_data.iterrows()]
    pd.Series(lls).to_csv(out_path, sep="\t", index=False, header=False)
    return out_path


def get_likelihood_dict(combined_df):
    ld = {}
    idx_list = list(combined_df.index)
    for i, key in enumerate(idx_list):
        if i % 2 == 1:
            continue
        parts = key.split("_")
        if len(parts) == 3:
            continue
        motif = parts[1]
        ld.setdefault(motif, {"true": [], "shuffled": []})
        ld[motif]["true"].append(combined_df.loc[key, "ll"])
        ld[motif]["shuffled"].append(combined_df.loc[idx_list[i + 1], "ll"])
    return ld


def compute_accuracy(ld):
    accs = {}
    for motif, d in ld.items():
        n = len(d["true"])
        accs[motif] = float(np.mean([d["true"][x] >= d["shuffled"][x] for x in range(n)]))
    return pd.DataFrame([accs], index=["Accuracy"]).T


def compute_and_save_metrics(all_seq_data, scores, model_name):
    lik_path = build_likelihood_tsv(scores, all_seq_data, model_name)
    
    lik_data = pd.read_csv(lik_path, sep="\t", header=None, names=["ll"])
    lik_data.index = all_seq_data["key"].values
    combined_df = pd.concat([all_seq_data.set_index("key"), lik_data], axis=1)
    ld = get_likelihood_dict(combined_df)
    lik_df = compute_accuracy(ld)
    
    lik_out = os.path.join(RES_DIR, f"{model_safe(model_name)}_accuracy.csv")
    lik_df.to_csv(lik_out)
    
    print(f"\nAccuracy Results:")
    print(lik_df.to_string())
    print(f"\nSaved to: {lik_out}")
    
    return lik_df


def main():
    np.random.seed(0)
    random.seed(0)
    
    if SELECTED_MODEL not in MODELS:
        print(f"Error: '{SELECTED_MODEL}' not in MODELS")
        print(f"Available: {list(MODELS.keys())}")
        return
    
    print(f"\nEvaluating: {SELECTED_MODEL}\n")
    
    ctrl_seqs = extract_ctrl_seqs_from_h5(DATA_H5, n=N_CTRL_SEQS)
    motif_dict = read_meme(MEME_FILE, motif_offset=MOTIF_OFFSET, max_motifs=N_MOTIFS)
    
    np.random.seed(0)
    random.seed(0)
    seq_dict = compile_footprint_seqs(ctrl_seqs, motif_dict)
    
    session_file = os.path.join(WORK_DIR, f"session_{MOTIF_OFFSET}_{MOTIF_OFFSET + N_MOTIFS}.txt")
    with open(session_file, "w") as f:
        for k, v in seq_dict.items():
            f.write(f"{k}\t{v}\n")
    
    seq_data = pd.read_csv(session_file, sep="\t", header=None, names=["key", "seq"], usecols=[0, 1]).reset_index(drop=True)
    
    model_cfg = MODELS[SELECTED_MODEL]
    model_type = model_cfg.get("type", "api")
    
    if model_type == "api":
        api_key = load_api_key(model_cfg["secret"])
        if not api_key:
            print("Error: No API key")
            return
        client = OpenAI(api_key=api_key, base_url=model_cfg["base_url"])
        scorer = make_api_scorer(client, model_cfg["model_id"], model_cfg["has_reasoning"])
        is_local = False
    else:
        scorer = make_local_scorer(model_cfg)
        is_local = True
    
    scores = score_all_sequences(seq_data, scorer, SELECTED_MODEL, is_local=is_local)
    
    all_motifs = read_meme(MEME_FILE, motif_offset=0, max_motifs=MOTIF_OFFSET + N_MOTIFS)
    np.random.seed(0)
    random.seed(0)
    all_seq_dict = compile_footprint_seqs(ctrl_seqs, all_motifs)
    scored_dict = {k: v for k, v in all_seq_dict.items() if k in scores}
    
    all_seq_file = os.path.join(WORK_DIR, f"all_scored_{model_safe(SELECTED_MODEL)}.txt")
    with open(all_seq_file, "w") as f:
        for k, v in scored_dict.items():
            f.write(f"{k}\t{v}\n")
    
    all_seq_data = pd.read_csv(all_seq_file, sep="\t", header=None, names=["key", "seq"], usecols=[0, 1]).reset_index(drop=True)
    
    compute_and_save_metrics(all_seq_data, scores, SELECTED_MODEL)
    
    if is_local and torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("\nDone\n")


if __name__ == "__main__":
    main()